# Chapter 29 — One Process, End to End

**Companion to *Applied AI*.**

Twenty-eight chapters of parts. This notebook walks one real unit of work —
reviewing a paragraph — through the preserved capstone run, then audits the
whole machine: thirteen joints, each classified by what the runtime actually
does about a violation, not by what the diagram suggests.

## Question

**Which joints are enforced, and which are only reconstructable?**

## What this notebook does

It **inspects** `capstone-composition/` (`report.json`'s twelve questions,
`results.json`'s happy path and hostile branches, the ledger's event kinds):
one task traced from request to completion — then the thirteen-joint audit
table, with the five unenforced joints named and the hostile cases that
exposed them.

```text
EXERCISED  ≠  REPRESENTED  ≠  NOT EXERCISED  ≠  NOT IMPLEMENTED
```

## Setup

Standard library only. No network, no API key, no `codeai` import. Only
bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="capstone-composition"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
CAP = EVIDENCE_DIR / "capstone-composition"
print("bundle: capstone-composition/")
report = json.loads((CAP / "report.json").read_text(encoding="utf-8"))
results = json.loads((CAP / "results.json").read_text(encoding="utf-8"))
print("report questions:", len(report))

bundle: capstone-composition/
report questions: 12


## 1. One paragraph, one process

Task `t-comp`: review the paragraph. The capstone report answers twelve
questions about that single unit of work — each answer pointing at the
durable record, not at prose. Read them in pipeline order:

In [2]:
for q in ["What was requested?",
           "What grant authorized delegation?",
           "What context was selected?",
           "Which model call ran, and what else did not?",
           "What did the worker report vs observe?",
           "What exact state was verified?",
           "Why accept?",
           "What did repetition do?",
           "What stayed unresolved?",
           "Why COMPLETE?",
           "What would the scheduler choose next?"]:
    print(q)
    print("  -> " + report[q])
    print()

What was requested?
  -> review the paragraph (task t-comp under child directive d-comp-edit)

What grant authorized delegation?
  -> child d-comp-edit narrows d-comp (causation 1d22a340...)

What context was selected?
  -> package d4dee3f436d6543d14a805b6e130ff2285f8b46911d3e59b6d58d7c72e038744 (trace 6b742ba1fec4aa1a7f63c381415fc91b49731b5b412ae37639d7b45d1a80764e)

Which model call ran, and what else did not?
  -> one fake call call-comp; checks/scheduler/authority deterministic; no model router

What did the worker report vs observe?
  -> reported succeeded; runtime observed 77e257264570... (before 72bfd9f4d550...)

What exact state was verified?
  -> target == observed == 77e257264570...; stale hash refused with 0 invocations

Why accept?
  -> bound PASS on exact bytes criteria + ACCEPT authority; causation-linked completion

What did repetition do?
  -> exact duplicate replayed (1 effect); changed instruction conflicted (0 effects)

What stayed unresolved?
  -> crash-gap retry sa

## 2. The ledger underneath

The happy path plus the hostile branches, as event kinds counted from the
preserved ledger. One fake model call (`model_calls: 1`); everything else —
checks, scheduler, authority — deterministic:

In [3]:
kinds = results["ledger_kinds"]
for kind in sorted(kinds):
    print(f"  {kinds[kind]:>2}x  {kind}")
print()
print("model calls:", results["model_calls"])
print("deterministic checks:", results["deterministic_checks"])
print("human gates:", results["human_gates"])
h = results["happy"]
print()
print("happy path: completion =", h["completion"],
      "| scheduler mid-run =", h["scheduler_mid"],
      "| first attempt =", h["denied_first"],
      "| replay reused =", h["replay_reused"],
      "| edit adapter calls =", h["edit_calls"])
assert h["completion"] == "completed" and h["edit_calls"] == 1
assert results["model_calls"] == 1
print()
print("One fake call carried the cognition; the joints carried the process.")

   6x  action.completed
   1x  action.replay_refused
   7x  action.requested
   1x  attempt.completed
   1x  attempt.interpreted
   1x  attempt.retry_decided
   1x  attempt.started
   1x  call.completed
   1x  call.manifest
   1x  call.requested
   1x  call.status_decided
   5x  check.completed
   5x  check.requested
   1x  claim.recorded
   1x  context.compilation_requested
   1x  context.compiled
   2x  directive.opened
   1x  task.accepted
   1x  task.completed
   3x  task.created

model calls: 1
deterministic checks: ['check-comp-1', 'check-comp-2', 'check-stale', 'check-fail-1']
human gates: ['requested_by human-approver (WRITE)', 'acceptance human-reviewer (ACCEPT)']

happy path: completion = completed | scheduler mid-run = CHECK | first attempt = denied | replay reused = act-comp-1 | edit adapter calls = 1

One fake call carried the cognition; the joints carried the process.


## 3. The branches that stop

The composition is not a golden path. Denied stays denied with the file
unchanged; a failed check completes nothing; a stale binding adds zero
invocations; a key collision on a changed instruction is refused; the crash
gap stays UNRESOLVED — FAILED cached, effect real:

In [4]:
print("denied path     :", results["denied_path"])
assert results["denied_path"] == {"status": "denied", "file_unchanged": True}
print("verify-fail path:", results["verify_fail_path"])
assert results["verify_fail_path"]["accepted"] is False
print("stale binding   :", results["stale"]["verdict"],
      "| added invocations:", results["stale"]["added_invocations"])
assert results["stale"]["verdict"] == "ERROR" and results["stale"]["added_invocations"] == 0
print("key collision   : raised =", results["collision"]["raised"],
      "| dimension:", results["collision"]["dimension"])
assert results["collision"]["raised"] is True
print("crash gap       :", results["crash_gap"]["first"], "->",
      results["crash_gap"]["retry"], "|", results["crash_gap"]["resolution"])
assert results["crash_gap"]["resolution"] == "UNRESOLVED"
print()
print("Every earlier chapter's refusal, firing inside one composed run.")

denied path     : {'status': 'denied', 'file_unchanged': True}
verify-fail path: {'check': 'FAIL', 'completion_without_acceptance': 'incomplete', 'accepted': False}
stale binding   : ERROR | added invocations: 0
key collision   : raised = True | dimension: instruction
crash gap       : failed -> failed | UNRESOLVED

Every earlier chapter's refusal, firing inside one composed run.


## 4. The thirteen-joint audit

A second process could *explain* every transition from the durable record.
The audit asked the harder question per joint — enforced, derived, recorded,
conventional, or absent? — each with a deliberately hostile case. Baseline
classifications from the frozen audit (Chapter 29); the notebook's evidence
for the weak ones is the hostile output beside each row:

In [5]:
joints = [
    ("directive -> action authority",        "Ch 20",    "ENFORCED",     "forged grant changes nothing"),
    ("directive -> acceptance authority",    "Ch 14, 20","RECORDED",     "caller-passed ACCEPT completed the task"),
    ("decision -> execution",                "Ch 28",    "CONVENTIONAL", "CHECK decided; a write merely coexisted"),
    ("action request -> effect",             "Ch 19",    "RECORDED",     "SUCCEEDED read as OBSERVED, pre == post"),
    ("effect -> recovery / reconciliation",  "Ch 16, 22","ENFORCED",     "crash gap projects unknown, not nothing"),
    ("observed state -> verification binding","Ch 21",   "ENFORCED",     "stale hash refused, 0 invocations"),
    ("command -> verdict semantics",         "Ch 21",    "ENFORCED",     "raising verifier -> ERROR, never PASS"),
    ("verification -> claim linkage",        "Ch 18, 21","DERIVED",      "ERROR left no claim-side record"),
    ("verification -> acceptance",           "Ch 14, 21","RECORDED",     "sys.exit(0) + artifact name accepted"),
    ("acceptance -> completion",             "Ch 14",    "ENFORCED",     "no validated acceptance, no completion"),
    ("replay -> current authority",          "Ch 22",    "ENFORCED",     "narrowed grant denies the replay"),
    ("durable state -> next operation",      "Ch 28",    "DERIVED",      "reconstructable, not a gate"),
    ("check -> artifact identity",           "Ch 14, 21","CONVENTIONAL", "caller-written target string trusted"),
]
print(f"{'joint':<42}{'built':<10}{'baseline':<13}hostile case")
print("-" * 110)
for name, built, cls, hostile in joints:
    print(f"{name:<42}{built:<10}{cls:<13}{hostile}")

from collections import Counter
tally = Counter(c for _, _, c, _ in joints)
print()
print("baseline tally:", dict(tally))
assert tally == {"ENFORCED": 6, "RECORDED": 3, "DERIVED": 2, "CONVENTIONAL": 2}
weak = sum(1 for _, _, c, _ in joints if c in ("RECORDED", "CONVENTIONAL"))
print("recorded/conventional (weaker than enforced):", weak)
assert weak == 5
print("Five joints weaker than their diagrams — plus one DERIVED gap (an ERROR")
print("with no claim-side record) and one DERIVED that is fine by design.")
print("The reconstruction looked complete; the audit showed which arrows only")
print("looked real.")

joint                                     built     baseline     hostile case
--------------------------------------------------------------------------------------------------------------
directive -> action authority             Ch 20     ENFORCED     forged grant changes nothing
directive -> acceptance authority         Ch 14, 20 RECORDED     caller-passed ACCEPT completed the task
decision -> execution                     Ch 28     CONVENTIONAL CHECK decided; a write merely coexisted
action request -> effect                  Ch 19     RECORDED     SUCCEEDED read as OBSERVED, pre == post
effect -> recovery / reconciliation       Ch 16, 22 ENFORCED     crash gap projects unknown, not nothing
observed state -> verification binding    Ch 21     ENFORCED     stale hash refused, 0 invocations
command -> verdict semantics              Ch 21     ENFORCED     raising verifier -> ERROR, never PASS
verification -> claim linkage             Ch 18, 21 DERIVED      ERROR left no claim-side recor

## 5. The finding nobody predicted

The most important weak joint was not among the five registered predictions.
The runtime held the before-reading and the after-reading, recorded both —
and never compared them. A worker that changed nothing and reported
`SUCCEEDED` completed "with the runtime's own observation of the resulting
state": an observation it never used. `FAILED` and `DENIED` were handled
correctly; the lie fit exactly the one shape the isolated tests never
checked — what two components *together* implied.

The repairs (each on its own branch, frozen baseline re-run after each) made
report, observation and verification three answers instead of one; resolved
acceptance authority from the task's recorded directive; bound checks to
store-resolved bytes; carried the selecting decision on governed operations;
and made every verification attempt navigable from its claim. Seven arrows
moved; seven did not — no repair disturbed a joint it was not aimed at.

## Interpretation

1. **Reconstructable ≠ enforced.** Explaining every transition from the
   record is necessary and not sufficient. The audit's taxonomy — enforced,
   derived, recorded, conventional, absent — is the transferable tool.
2. **Test joints, not just components.** Isolated tests passed because each
   asserted what its component reported. The failures lived between
   components.
3. **A successful composition can still lie.** The happy path completed while
   five joints were habits, not guarantees. The notebook audits the system
   rather than staging a perfect ending.
4. **Preserved limits.** The reopen "across a process boundary" projects
   completion on the same live handle — the bundle shows a verifier reading
   a checkpointed ledger, not a fresh handle. And no repair settles who was
   entitled to name the human in an authority transition.

## Try it yourself

1. Open `ledger.sqlite`: count `action.requested` (7) vs `action.completed`
   (6). Which request has no completion — and what does the record say
   instead?
2. The crash gap resolves UNRESOLVED. Sketch the reconciliation record you
   would want before a human re-runs the effect — which three facts must it
   join?
3. Pick one CONVENTIONAL joint. Write the hostile case from the audit as an
   assertion against current CodeAI. Does it still fail?

*Evidence: `experiments/applied-ai/evidence/capstone-composition/`
(`report.json`, `results.json`, ledger + artifacts + stdlib verifier). No
network, no API key, no `codeai` import.*